Reproduction du notebook brainbeats_sub005_allruns

1. Préparation des données

1.1 Chargement des bibliothèques

In [1]:
import os
import nibabel as nib
import pandas as pd
import numpy as np
from nilearn.input_data import NiftiLabelsMasker
from nilearn import datasets
from tqdm import tqdm  # progress bar

/tmp/ipykernel_2419/2839967032.py:5: FutureWarning: The import path 'nilearn.input_data' is deprecated in version 0.9. Importing from 'nilearn.input_data' will be possible at least until release 0.13.0. Please import from 'nilearn.maskers' instead.
  from nilearn.input_data import NiftiLabelsMasker


1.2 Définition des chemins d'accès

In [2]:
# Set up
base_path = "./ds003720" #auparavant «base_path = "."»
subject = "sub-005"
n_runs = 6
t_r = 2.0  # repetition time (2 seconds)

Comme le notebook brainbeats_analysis, j'ai dû changer le fichier d'accès aux données

1.3 Téléchargement de l'atlas et préparation du masque

In [3]:
# Load atlas & prepare masker
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100, resolution_mm=2)
masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize=True, t_r=t_r)

# Prepare result lists
X_all = []
y_all = []

[fetch_atlas_schaefer_2018] Dataset found in /home/etudiants/nilearn_data/schaefer_2018

2. Vérification de la présence des fichiers et téléchargement des données

In [5]:
# Loop through all 6 runs
for run in tqdm(range(1, n_runs + 1)):
    run_str = f"run-0{run}"
    bold_file = f"{base_path}/{subject}/func/{subject}_task-Test_{run_str}_bold.nii"
    events_file = f"{base_path}/{subject}/func/{subject}_task-Test_{run_str}_events.tsv"

    # Check if both files exist
    
    print(f"Looking for: {bold_file}")
    print(f"Looking for: {events_file}")


    if not os.path.exists(bold_file) or not os.path.exists(events_file):
        print(f"Missing: {run_str}")
        continue

    # Load data
    func_img = nib.load(bold_file)
    events = pd.read_csv(events_file, sep="\t")
    roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]

    # Segment into trials
    for _, row in events.iterrows():
        onset = row['onset']
        duration = row['duration']
        genre = row['genre'].strip("'").strip('"')

        start_vol = int(onset / t_r)
        end_vol = int((onset + duration) / t_r)
        trial_ts = roi_ts[start_vol:end_vol, :]

        if trial_ts.shape[0] < 2:
            continue  # skip too-short segments

        # Compute connectivity
        conn_matrix = np.corrcoef(trial_ts.T)

        # Flatten upper triangle
        try:
            flat = conn_matrix[np.triu_indices_from(conn_matrix, k=1)]
            if not np.isnan(flat).any() and len(flat) == 4950:
                X_all.append(flat)
                y_all.append(genre)
        except:
            print(f"⚠️ Skipped a trial due to shape or NaN issue.")


  0%|                                                                                 | 0/6 [00:00<?, ?it/s]

Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-01_bold.nii
Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-01_events.tsv


/tmp/ipykernel_2419/3190822257.py:20: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
 17%|████████████▏                                                            | 1/6 [00:17<01:27, 17.52s/it]

Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-02_bold.nii
Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-02_events.tsv


/tmp/ipykernel_2419/3190822257.py:20: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
 33%|████████████████████████▎                                                | 2/6 [00:31<01:02, 15.73s/it]

...

/tmp/ipykernel_2419/3190822257.py:20: UserWarning: After resampling the label image to the data image, the following labels were removed: {np.float32(4.0), np.float32(5.0), np.float32(54.0)}. Label image only contains 98 labels (including background).
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
/tmp/ipykernel_2419/3190822257.py:20: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
 67%|████████████████████████████████████████████████▋                        | 4/6 [01:05<00:33, 16.53s/it]

Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-05_bold.nii
Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-05_events.tsv


/tmp/ipykernel_2419/3190822257.py:20: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
 83%|████████████████████████████████████████████████████████████▊            | 5/6 [01:22<00:16, 16.78s/it]

Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-06_bold.nii
Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-06_events.tsv


/tmp/ipykernel_2419/3190822257.py:20: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
100%|█████████████████████████████████████████████████████████████████████████| 6/6 [01:40<00:00, 16.73s/it]


Comme le notebook brainbeats_analysis, j'ai dû corriger les fichier bold et events en ajoutant «/func/{subject}»

3. Convertir en liste numpy et sauvegarder des fichier pour l'utiliser dans le script de visualisation

In [6]:
# Convert to numpy arrays
X_all = np.array(X_all)
y_all = np.array(y_all)

print("✅ All runs processed!")
print("Total trials:", X_all.shape[0])
print("Feature shape per trial:", X_all.shape[1])
print("Unique genres:", np.unique(y_all))

# 💾 Save X_all for later visualization use
np.save("X_all.npy", X_all)

✅ All runs processed!
Total trials: 164
Feature shape per trial: 4950
Unique genres: ['blues' 'classical' 'country' 'disco' 'hiphop' 'jazz' 'metal' 'pop'
 'reggae' 'rock']
